# 2. Predicting eCLIP signal

This notebook runs the model on real sequence and reads what comes out: three tracks per
task, each a density over positions.

The point to internalise is that PARNET predicts **shape, not depth**. Every track is a
probability distribution over the window, so it tells you where signal falls, not how
much of it there is. Comparing a prediction to data therefore means comparing shapes, and
the right statistic is a correlation over positions rather than an error in read counts.

In [ ]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt
from parnet.models import RBPNet
from parnet.tokenization import RnaTokenizer

model = RBPNet.from_pretrained(pt.model_path('parnet-21m-penalty5'))
task_names = pt.task_names('encode-223')
print('tasks:', len(task_names))

## From sequence to prediction

The tokenizer accepts DNA or RNA, upper or lower case, and is explicit about everything
else. This matters more than it sounds: the old helper silently turned a lowercase
sequence into an all-zero tensor and the model still returned a confident prediction.
Soft-masked genome sequence is lowercase, so that bug was reachable from ordinary input.

In [ ]:
tokenizer = RnaTokenizer()

for seq in ['ACGT' * 4, 'acgt' * 4, 'ACGU' * 4]:
    encoded = tokenizer(seq)
    ids = encoded['input_ids']
    print(f'{seq[:8]:10s} -> ids {ids[0, :8].tolist()}  (sum {int(ids.sum())})')

print()
print('DNA and RNA and case all give the same ids:',
      torch.equal(tokenizer('ACGT')['input_ids'], tokenizer('acgu')['input_ids']))

In [ ]:
rng = np.random.default_rng(0)
sequence = ''.join(rng.choice(list('ACGT'), size=600))

with torch.no_grad():
    out = model(**tokenizer(sequence))

for key, value in out.items():
    print(f'{key:12s} {tuple(value.shape)}')

`target`, `control` and `total` are log-densities over the 600 positions, for each of the
223 tasks. Exponentiating gives a distribution that sums to 1.

In [ ]:
probs = out['total'].exp()
sums = probs.sum(-1)
print('per-task sums: min %.6f max %.6f' % (sums.min(), sums.max()))
print()
print('So a "prediction" is a shape. There is no read-count scale in it,')
print('which is why the training loss is multinomial.')

## Reading one task

Pick a task by name rather than by index. Indices are a property of the dataset the model
was trained on and are easy to get wrong.

In [ ]:
rbp = 'RBFOX2_K562'
idx = task_names.index(rbp)
print(f'{rbp} is task index {idx}')

target = out['target'][0, idx].exp().numpy()
control = out['control'][0, idx].exp().numpy()
total = out['total'][0, idx].exp().numpy()

print(f'target  peak at position {target.argmax():3d}, height {target.max():.5f}')
print(f'control peak at position {control.argmax():3d}, height {control.max():.5f}')
print(f'total   peak at position {total.argmax():3d}, height {total.max():.5f}')

w = torch.sigmoid(out['mix_coeff'][0, idx]).item()
print(f'\ntarget weight w for this task: {w:.4f}')
print('total is dominated by', 'target' if w > 0.5 else 'control')

## Plot it

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(target, label='target', linewidth=1.0)
ax.plot(control, label='control', linewidth=1.0)
ax.plot(total, label='total', linewidth=1.4, color='black')
ax.set_xlabel('position in window (nt)')
ax.set_ylabel('predicted density')
ax.set_title(f'{rbp}, random sequence, w = {w:.3f}')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig('02_profile.png', dpi=120)
print('saved 02_profile.png')

## Batching

Windows are independent, so batching is free parallelism and the model is fully
convolutional, so a batch is one call.

Two things worth checking rather than assuming: that the answer does not depend on batch
size, and that the output shapes do not either. The second was not true until recently:
`SequenceLinearMix` squeezed every size-1 axis, so at batch size 1 the batch axis
vanished and `mix_coeff` came back with shape `(tasks,)` instead of `(1, tasks)`.

In [ ]:
import time

sequences = [''.join(rng.choice(list('ACGT'), size=600)) for _ in range(16)]
encoded = tokenizer(sequences)
print('batched ids:', tuple(encoded['input_ids'].shape))

started = time.time()
with torch.no_grad():
    batched = model(**encoded)
elapsed = time.time() - started
print(f'{len(sequences)} windows in {elapsed:.2f} s on CPU '
      f'({elapsed / len(sequences) * 1000:.0f} ms per window)')
print('output:', tuple(batched['total'].shape))

# Does batching change the answer? Only by float32 round-off: convolutions pick
# different kernels at different batch sizes. Worth measuring rather than assuming.
with torch.no_grad():
    alone = model(**tokenizer(sequences[0]))

difference = (alone['total'] - batched['total'][:1]).abs()
scale = batched['total'][:1].abs().max()
print()
print(f'alone vs in-batch: max abs diff {difference.max():.2e} on values up to {scale:.1f}')
print(f'relative: {difference.max() / scale:.1e}')

profile_alone = alone['total'][0, 0].exp().numpy()
profile_batch = batched['total'][0, 0].exp().numpy()
print(f'profile correlation: {np.corrcoef(profile_alone, profile_batch)[0, 1]:.10f}')
print()
print('Shapes do not depend on batch size either:')
print('  batch of 1 :', tuple(alone['mix_coeff'].shape))
print('  batch of 16:', tuple(batched['mix_coeff'].shape))

## Comparing to data: use a correlation, not an error

Because predictions are shapes, the per-window statistic that means something is the
Pearson correlation between the predicted density and the observed read profile over
positions. Windows with no reads carry no shape at all and must be dropped before
averaging, or they dilute the result with noise.

The helper below is the whole evaluation idea in eight lines.

In [ ]:
def profile_pcc(predicted, observed, min_reads=10):
    """Pearson correlation over positions, for windows with enough signal to have a shape.

    Returns NaN for windows below min_reads rather than a misleading number. Averaging
    over NaN-free entries only is the caller's job, and is deliberate: silently treating
    an empty window as correlation 0 is how an evaluation gets quietly pessimistic.
    """
    observed = np.asarray(observed, dtype=float)
    if observed.sum() < min_reads:
        return float('nan')
    predicted = np.asarray(predicted, dtype=float)
    if predicted.std() == 0 or observed.std() == 0:
        return float('nan')
    return float(np.corrcoef(predicted, observed)[0, 1])


# Demonstrate on synthetic observations: a window that follows the prediction, one that
# does not, and one with almost no reads.
follows = rng.poisson(target * 5000)
unrelated = rng.poisson(np.roll(target, 300) * 5000)
empty = rng.poisson(target * 0.5)

for label, observed in [('follows prediction', follows),
                        ('shifted by 300 nt', unrelated),
                        ('almost no reads', empty)]:
    value = profile_pcc(total, observed)
    print(f'{label:20s} reads {observed.sum():7.0f}  PCC {value:.3f}' if value == value
          else f'{label:20s} reads {observed.sum():7.0f}  dropped (below threshold)')

## What to take away

1. Predictions are densities over positions. There is no depth in them.
2. Address tasks by name; indices belong to the dataset.
3. The tokenizer handles case and DNA/RNA explicitly, so soft-masked sequence is safe.
4. Evaluate with a per-window correlation, and drop windows with too few reads rather
   than scoring them as zero.